## 1. Load the data
Loading the 2014 OSMI Mental Health in Tech survey and getting a first look at its size, columns, and data types.

In [ ]:
import pandas as pd
import numpy as np


df = pd.read_csv("../data/raw/survey.csv")

print(df.shape)
df.info()
df.head()

(1259, 27)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1259 entries, 0 to 1258
Data columns (total 27 columns):
 #   Column                     Non-Null Count  Dtype 
---  ------                     --------------  ----- 
 0   Timestamp                  1259 non-null   object
 1   Age                        1259 non-null   int64 
 2   Gender                     1259 non-null   object
 3   Country                    1259 non-null   object
 4   state                      744 non-null    object
 5   self_employed              1241 non-null   object
 6   family_history             1259 non-null   object
 7   treatment                  1259 non-null   object
 8   work_interfere             995 non-null    object
 9   no_employees               1259 non-null   object
 10  remote_work                1259 non-null   object
 11  tech_company               1259 non-null   object
 12  benefits                   1259 non-null   object
 13  care_options               1259 non-null   object
 1

,Timestamp,Age,Gender,Country,state,self_employed,family_history,treatment,work_interfere,no_employees,...,leave,mental_health_consequence,phys_health_consequence,coworkers,supervisor,mental_health_interview,phys_health_interview,mental_vs_physical,obs_consequence,comments
0,2014-08-27 11:29:31,37,Female,United States,IL,NaN,No,Yes,Often,6-25,...,Somewhat easy,No,No,Some of them,Yes,No,Maybe,Yes,No,NaN
1,2014-08-27 11:29:37,44,M,United States,IN,NaN,No,No,Rarely,More than 1000,...,Don't know,Maybe,No,No,No,No,No,Don't know,No,NaN
2,2014-08-27 11:29:44,32,Male,Canada,NaN,NaN,No,No,Rarely,6-25,...,Somewhat difficult,No,No,Yes,Yes,Yes,Yes,No,No,NaN
3,2014-08-27 11:29:46,31,Male,United Kingdom,NaN,NaN,Yes,Yes,Often,26-100,...,Somewhat difficult,Yes,Yes,Some of them,No,Maybe,Maybe,No,Yes,NaN
4,2014-08-27 11:30:22,31,Male,United States,TX,NaN,No,No,Never,100-500,...,Don't know,No,No,Some of them,Yes,Yes,Yes,Don't know,No,NaN


## 2. Missing values
Checking which columns have missing data and how much, to decide how to handle each one.

In [9]:
missing = df.isna().sum()
missing_pct = (missing / len(df) * 100).round(1)

pd.DataFrame({"missing": missing, "percent": missing_pct})[missing > 0].sort_values("missing", ascending=False)

,missing,percent
comments,1095,87.0
state,515,40.9
work_interfere,264,21.0
self_employed,18,1.4


**Findings:** `comments` is mostly empty (optional free text). `state` is likely only filled in for US respondents. `work_interfere` blanks probably mean the respondent had no condition. `self_employed` is missing for 18 rows. Each is handled in its own section below.

## 3. Age
Checking that ages fall in a realistic range for working adults.

In [10]:
df["Age"].describe()

count    1.259000e+03
mean     7.942815e+07
std      2.818299e+09
min     -1.726000e+03
25%      2.700000e+01
50%      3.100000e+01
75%      3.600000e+01
max      1.000000e+11
Name: Age, dtype: float64

In [11]:
df[(df["Age"] < 18) | (df["Age"] > 75)]["Age"].sort_values()

715           -1726
143             -29
1127             -1
734               5
989               8
1090             11
364             329
390     99999999999
Name: Age, dtype: int64

In [14]:
# Treat anything outside 18–75 as a bad entry
bad_age = (df["Age"] < 18) | (df["Age"] > 75)
print(f"Invalid ages: {bad_age.sum()}")

df.loc[bad_age, "Age"] = np.nan
df["Age"] = df["Age"].fillna(df["Age"].median()).astype(int)

df["Age"].describe()

Invalid ages: 8


count    1259.000000
mean       32.069897
std         7.265565
min        18.000000
25%        27.000000
50%        31.000000
75%        36.000000
max        72.000000
Name: Age, dtype: float64

**Decision:** 8 ages were clearly invalid (negative, 5–11, 329, and 99999999999). Used 18–75 as a plausible range for working adults; no real ages fall near either cutoff, so the exact bounds don't change which rows are flagged. Replaced invalid ages with the median (31) to keep those respondents' other answers.

## 4. Gender
`Gender` was a free-text field, so the same answer appears in many forms. Looking at every distinct response before grouping them.

In [15]:
df["Gender"].value_counts()

Gender
Male                                              615
male                                              206
Female                                            121
M                                                 116
female                                             62
F                                                  38
m                                                  34
f                                                  15
Make                                                4
Male                                                3
Woman                                               3
Cis Male                                            2
Man                                                 2
Female (trans)                                      2
Female                                              2
Trans woman                                         1
msle                                                1
male leaning androgynous                            1
Neuter               

In [21]:
df["Gender"].str.strip().str.lower().value_counts().to_string()

'Gender\nmale                990\nfemale              251\nnon-binary/other     14\nunknown               4'

In [23]:
# Save the original answers only the first time, so re-running is safe
if "Gender_raw" not in df.columns:
    df["Gender_raw"] = df["Gender"]

g = df["Gender_raw"].str.strip().str.lower()

male = {"male", "m", "man", "cis male", "cis man", "male (cis)",
        "make", "mal", "maile", "mail", "malr", "msle"}
female = {"female", "f", "woman", "cis female", "female (cis)", "cis-female/femme",
          "femake", "femail", "female (trans)", "trans woman", "trans-female"}
unknown = {"a little about you", "p", "all", "nah"}

def group_gender(x):
    if x in male:
        return "Male"
    if x in female:
        return "Female"
    if x in unknown:
        return "Unknown"
    return "Non-binary/Other"

df["Gender"] = g.map(group_gender)

df["Gender"].value_counts()

Gender
Male                990
Female              251
Non-binary/Other     14
Unknown               4
Name: count, dtype: int64

## 5. Country and state
Checking country spread and whether `state` is only filled in for US respondents.

In [24]:
df["Country"].value_counts().head(15)

Country
United States     751
United Kingdom    185
Canada             72
Germany            45
Ireland            27
Netherlands        27
Australia          21
France             13
India              10
New Zealand         8
Poland              7
Switzerland         7
Sweden              7
Italy               7
South Africa        6
Name: count, dtype: int64

In [25]:
is_us = df["Country"] == "United States"
pd.crosstab(is_us, df["state"].isna(), rownames=["US?"], colnames=["state missing?"])

state missing?,False,True
US?,,
False,4,504
True,740,11
